In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.utils import resample

In [ ]:
# Phase 1

In [ ]:
# Define the file path to the dataset, the random seed, and the test set proportion

In [ ]:
path = r"D:\Articles\4\Data\Raw\ABIDEII_Composite_Phenotypic.csv"
random_state = 42
test_size = 0.30

In [ ]:
# Select the columns to keep, define SRS-related columns, load the dataset, and clean column names

In [ ]:

cols_keep = [
    "AGE_AT_SCAN", "SEX", "DX_GROUP",
    "SRS_TOTAL_RAW", "SRS_AWARENESS_RAW", "SRS_COGNITION_RAW",
    "SRS_COMMUNICATION_RAW", "SRS_MOTIVATION_RAW", "SRS_MANNERISMS_RAW"
]

srs_cols = [
    "SRS_TOTAL_RAW", "SRS_AWARENESS_RAW", "SRS_COGNITION_RAW",
    "SRS_COMMUNICATION_RAW", "SRS_MOTIVATION_RAW", "SRS_MANNERISMS_RAW"
]

Data_df = pd.read_csv(path, encoding="cp1252")
Data_df.columns = Data_df.columns.astype(str).str.strip()

In [ ]:
# Select needed columns, add a row ID, clean diagnosis values, filter valid groups, and convert sex to numeric

In [ ]:
df = Data_df.loc[:, cols_keep].copy()
df["row_id"] = df.index.astype(np.int64)

df["DX_GROUP"] = pd.to_numeric(df["DX_GROUP"], errors="coerce")
df = df[df["DX_GROUP"].isin([1, 2])].copy()
df["DX_GROUP"] = df["DX_GROUP"].astype(int)

df["SEX"] = pd.to_numeric(df["SEX"], errors="coerce")


In [ ]:
# Convert SRS columns to numeric values and split the data into stratified train and test sets

In [ ]:
for c in srs_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df_train, df_test = train_test_split(
    df,
    test_size=test_size,
    random_state=random_state,
    stratify=df["DX_GROUP"]
)

df_train = df_train.copy()
df_test = df_test.copy()

In [ ]:
# Compute group-wise mean SRS scores by diagnosis and overall mean SRS scores from the training data

In [ ]:
train_means = df_train.groupby("DX_GROUP", observed=True)[srs_cols].mean()
global_means = df_train[srs_cols].mean()

In [ ]:
# Fill missing SRS values using diagnosis-specific means, and use global means as a fallback

In [ ]:
def impute_groupwise_means(df_part: pd.DataFrame,
                           means_by_group: pd.DataFrame,
                           fallback: pd.Series) -> pd.DataFrame:
    out = df_part.copy()
    group_mean_df = means_by_group.reindex(out["DX_GROUP"].to_numpy()).set_index(out.index)

    for c in srs_cols:
        out[c] = out[c].fillna(group_mean_df[c])
        out[c] = out[c].fillna(fallback[c])

    return out

In [ ]:
# Apply group-wise mean imputation to the training and test datasets

In [ ]:
df_train_imp = impute_groupwise_means(df_train, train_means, global_means)
df_test_imp  = impute_groupwise_means(df_test,  train_means, global_means)

In [ ]:
# Compute min and max SRS values from training data and prepare safe denominators for scaling

In [ ]:
train_min = df_train_imp[srs_cols].min()
train_max = df_train_imp[srs_cols].max()
denom = (train_max - train_min).replace(0, np.nan)

In [ ]:
# Apply min–max scaling to SRS columns using training data statistics, with safe handling for zero ranges

In [ ]:
def minmax_apply(df_part: pd.DataFrame, mn: pd.Series, denom: pd.Series) -> pd.DataFrame:
    out = df_part.copy()
    for c in srs_cols:
        if pd.isna(denom[c]):
            out[c] = 0.0
        else:
            out[c] = (out[c] - mn[c]) / denom[c]
    return out

In [ ]:
# Scale SRS values in training and test data using min–max normalization

In [ ]:
df_train_scaled = minmax_apply(df_train_imp, train_min, denom)
df_test_scaled  = minmax_apply(df_test_imp,  train_min, denom)

In [ ]:
# Clip scaled SRS values to keep them within the 0 to 1 range

In [ ]:
df_test_scaled[srs_cols] = df_test_scaled[srs_cols].clip(0.0, 1.0)

In [ ]:
# Collect imputation means and scaling parameters for each SRS column and diagnosis group as metadata

In [ ]:
meta_rows = []
means_dict = train_means.to_dict(orient="index")
for c in srs_cols:
    for g in [1, 2]:
        val = np.nan
        gd = means_dict.get(g)
        if gd is not None:
            v = gd.get(c)
            val = float(v) if v is not None and pd.notna(v) else np.nan
        meta_rows.append({"param": "impute_mean", "DX_GROUP": g, "column": c, "value": val})

    meta_rows.append({"param": "train_min", "DX_GROUP": np.nan, "column": c, "value": float(train_min[c]) if pd.notna(train_min[c]) else np.nan})
    meta_rows.append({"param": "train_max", "DX_GROUP": np.nan, "column": c, "value": float(train_max[c]) if pd.notna(train_max[c]) else np.nan})

In [ ]:
# Phase 2

In [ ]:
# Build three proxy values from SRS scores, keep them in a valid range,
# add a small tie-breaker noise, and assign a preferred arm label for each row

In [ ]:

def build_proxies(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "row_id" not in out.columns:
        out["row_id"] = np.arange(out.shape[0], dtype=np.int64)

    out["p_arm1"] = 1.0 - out["SRS_MOTIVATION_RAW"]
    out["p_arm2"] = 1.0 - out[["SRS_AWARENESS_RAW", "SRS_COGNITION_RAW", "SRS_COMMUNICATION_RAW"]].mean(axis=1)
    out["p_arm3"] = 1.0 - out["SRS_MANNERISMS_RAW"]

    p_cols = ["p_arm1", "p_arm2", "p_arm3"]
    out[p_cols] = out[p_cols].clip(0.0, 1.0)

    row = out["row_id"].astype(np.int64).to_numpy()
    base = out[p_cols].to_numpy(dtype=float)

    arm_idx = np.arange(3, dtype=np.int64)
    micro = ((row[:, None] * 1_315_423_911 + arm_idx[None, :] * 2_654_435_761) % 10_000) / 10_000.0

    best_idx = np.argmax(base + 1e-9 * micro, axis=1)
    out["preferred_arm"] = best_idx + 1

    arm_map = {1: "arm_motivation", 2: "arm_social_cognitive", 3: "arm_behavioral"}
    out["preferred_arm_label"] = out["preferred_arm"].map(arm_map)
    out["preferred_arm_col"] = np.array(p_cols, dtype=object)[best_idx]

    return out

In [ ]:
# Generate proxy values and preferred arm labels for the scaled training and test data

In [ ]:
df_train_p2 = build_proxies(df_train_scaled)
df_test_p2  = build_proxies(df_test_scaled)

In [ ]:
# Phase 3

In [ ]:
# Define context features, proxy columns, and the final set of columns to keep

In [ ]:
context_cols = ["AGE_AT_SCAN", "SEX", "DX_GROUP"]
p_cols = ["p_arm1", "p_arm2", "p_arm3"]
keep_cols = ["row_id"] + context_cols + p_cols + ["preferred_arm"]

In [ ]:
# Keep only required columns, convert types, drop rows with missing values,
# validate preferred_arm and row_id, and clip proxy values to 0–1

In [ ]:
def make_context_ready(df: pd.DataFrame) -> pd.DataFrame:
    out = df.loc[:, keep_cols].copy()

    out["AGE_AT_SCAN"] = pd.to_numeric(out["AGE_AT_SCAN"], errors="coerce")
    out["SEX"] = pd.to_numeric(out["SEX"], errors="coerce")
    out["DX_GROUP"] = pd.to_numeric(out["DX_GROUP"], errors="coerce")
    out["preferred_arm"] = pd.to_numeric(out["preferred_arm"], errors="coerce")

    for c in p_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out = out.dropna(subset=keep_cols).copy()

    out["row_id"] = out["row_id"].astype(np.int64)
    out["AGE_AT_SCAN"] = out["AGE_AT_SCAN"].astype(float)
    out["SEX"] = out["SEX"].astype(int)
    out["DX_GROUP"] = out["DX_GROUP"].astype(int)
    out["preferred_arm"] = out["preferred_arm"].astype(int)

    if not out["preferred_arm"].isin([1, 2, 3]).all():
        raise ValueError("preferred_arm has unexpected values")

    out[p_cols] = out[p_cols].clip(0.0, 1.0)

    if out["row_id"].duplicated().any():
        raise ValueError("row_id is not unique after cleaning")

    return out

In [ ]:
# Finalize and validate training and test data so they are ready for modeling

In [ ]:
df_train_p3 = make_context_ready(df_train_p2)
df_test_p3  = make_context_ready(df_test_p2)

In [ ]:
# Check that training and test sets have no overlapping row IDs

In [ ]:
if np.intersect1d(df_train_p3["row_id"].to_numpy(), df_test_p3["row_id"].to_numpy()).size:
    raise ValueError("row_id overlap between train and test")

In [ ]:
# Phase 4

In [ ]:
# Define context feature columns and proxy value columns

In [ ]:
context_cols = ["AGE_AT_SCAN", "SEX", "DX_GROUP"]
p_cols = ["p_arm1", "p_arm2", "p_arm3"]

In [ ]:
# Set the number of available arms (options)

In [ ]:
K = 3

In [ ]:
# Extract training context features and target preferred arm labels

In [ ]:
X_train = df_train_p3[context_cols].copy()
y_train = df_train_p3["preferred_arm"].copy()

In [ ]:
# Extract test context features and labels, and store the test sample size


In [ ]:
X_test = df_test_p3[context_cols].copy()
y_test = df_test_p3["preferred_arm"].copy()

N_test = X_test.shape[0]

In [ ]:
# Compute a stable row-wise softmax to convert scores into probability distributions

In [ ]:
def softmax_rowwise(scores: np.ndarray) -> np.ndarray:
    z = scores - np.max(scores, axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / np.clip(ez.sum(axis=1, keepdims=True), 1e-12, None)

In [ ]:
# Normalize each row so values are non-negative and sum to one

In [ ]:
def normalize_rows(pi: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    pi = np.clip(pi, 0.0, None)
    denom = np.clip(pi.sum(axis=1, keepdims=True), eps, None)
    return pi / denom

In [ ]:
# Define all possible class labels for the arms

In [ ]:
ALL_CLASSES = np.array([1, 2, 3], dtype=int)

In [ ]:
# Predict probabilities and align them to a fixed class order for all samples

In [ ]:
def predict_proba_full(model: LogisticRegression, X: pd.DataFrame, all_classes: np.ndarray) -> np.ndarray:
    proba = model.predict_proba(X)
    out = np.zeros((X.shape[0], len(all_classes)), dtype=float)
    for j, cls in enumerate(model.classes_):
        k = int(np.where(all_classes == int(cls))[0][0])
        out[:, k] = proba[:, j]
    return out

In [ ]:
# Initialize a multinomial logistic regression model with stable solver settings

In [ ]:
logreg = LogisticRegression(
    solver="lbfgs",
    max_iter=4000
)

In [ ]:
# Train the logistic regression model on the training data

In [ ]:
logreg.fit(X_train, y_train)

In [ ]:
# Predict class probabilities on the test data and normalize them to sum to one

In [ ]:
pi_logreg_test = normalize_rows(predict_proba_full(logreg, X_test, ALL_CLASSES))

In [ ]:
# Create a uniform random probability distribution across all arms for the test set

In [ ]:
pi_random_test = np.ones((N_test, K), dtype=float) / K

In [ ]:
# Set the epsilon value for probability mixing or smoothing

In [ ]:
epsilon = 0.1

In [ ]:
# Select the arm with the highest predicted probability for each test sample

In [ ]:
best_arm = np.argmax(pi_logreg_test, axis=1)

In [ ]:
# Initialize epsilon-smoothed probabilities with a small uniform value

In [ ]:
pi_eps_test = np.full((N_test, K), fill_value=epsilon / K, dtype=float)

In [ ]:
# Add most of the probability mass to the best predicted arm (epsilon-greedy)

In [ ]:
pi_eps_test[np.arange(N_test), best_arm] += (1.0 - epsilon)

In [ ]:
# Normalize epsilon-greedy probabilities so each row sums to one

In [ ]:
pi_eps_test = normalize_rows(pi_eps_test)

In [ ]:
# Set the number of bootstrap samples and allocate space for ensemble probabilities

In [ ]:
B = 100
pi_ens = np.zeros((B, N_test, K), dtype=float)

In [ ]:
# Bootstrap resampling and refitting to obtain normalized probability estimates on the test set.

In [ ]:
for b in range(B):
    Xb, yb = resample(X_train, y_train, replace=True, random_state=10_000 + b)
    m = LogisticRegression(
        solver="lbfgs",
        max_iter=4000
    )
    m.fit(Xb, yb)
    pi_ens[b] = normalize_rows(predict_proba_full(m, X_test, ALL_CLASSES))

In [ ]:
# Compute the mean and standard deviation of probabilities across bootstrap samples

In [ ]:
pi_mean = pi_ens.mean(axis=0)
pi_std  = pi_ens.std(axis=0)

In [ ]:
# Set the exploration strength parameter for the upper confidence bound (UCB)

In [ ]:
ucb_c = 2.0

In [ ]:
# Build UCB-based probabilities using mean and uncertainty, then normalize them

In [ ]:
pi_ucb_boot_test = normalize_rows(softmax_rowwise(pi_mean + ucb_c * pi_std))

In [ ]:
# Initialize a random number generator with a fixed seed

In [ ]:
rng = np.random.default_rng(123)

In [ ]:
# Randomly select one bootstrap index for each test sample

In [ ]:
chosen_b = rng.integers(0, B, size=N_test)

In [ ]:
# Select one bootstrap probability sample per test case and normalize it

In [ ]:
pi_boot_sample_test = normalize_rows(pi_ens[chosen_b, np.arange(N_test), :])

In [ ]:
# Prepare the final test DataFrame with row ID, context features, and true labels

In [ ]:
df_phase4_test = df_test_p3[["row_id"] + context_cols].copy()
df_phase4_test["preferred_arm"] = y_test.to_numpy()

In [ ]:
# Add uniform random policy probabilities for each arm to the test DataFrame

In [ ]:
df_phase4_test["pi_random_arm1"] = pi_random_test[:, 0]
df_phase4_test["pi_random_arm2"] = pi_random_test[:, 1]
df_phase4_test["pi_random_arm3"] = pi_random_test[:, 2]

In [ ]:
# Add logistic regression policy probabilities for each arm to the test DataFrame

In [ ]:
df_phase4_test["pi_logreg_arm1"] = pi_logreg_test[:, 0]
df_phase4_test["pi_logreg_arm2"] = pi_logreg_test[:, 1]
df_phase4_test["pi_logreg_arm3"] = pi_logreg_test[:, 2]

In [ ]:
# Add epsilon-greedy policy probabilities for each arm to the test DataFrame

In [ ]:
df_phase4_test["pi_eps_arm1"] = pi_eps_test[:, 0]
df_phase4_test["pi_eps_arm2"] = pi_eps_test[:, 1]
df_phase4_test["pi_eps_arm3"] = pi_eps_test[:, 2]

In [ ]:
# Add UCB bootstrap-based policy probabilities for each arm to the test DataFrame

In [ ]:
df_phase4_test["pi_ucb_boot_arm1"] = pi_ucb_boot_test[:, 0]
df_phase4_test["pi_ucb_boot_arm2"] = pi_ucb_boot_test[:, 1]
df_phase4_test["pi_ucb_boot_arm3"] = pi_ucb_boot_test[:, 2]

In [ ]:
# Add single bootstrap-sample policy probabilities for each arm to the test DataFrame

In [ ]:
df_phase4_test["pi_boot_sample_arm1"] = pi_boot_sample_test[:, 0]
df_phase4_test["pi_boot_sample_arm2"] = pi_boot_sample_test[:, 1]
df_phase4_test["pi_boot_sample_arm3"] = pi_boot_sample_test[:, 2]

In [ ]:
# Phase 5

In [ ]:
# Merge policy outputs with proxy values, remove duplicates, sort by row ID, and reset the index

In [ ]:
df = (
    df_phase4_test.merge(
       df_test_p3[["row_id", "p_arm1", "p_arm2", "p_arm3"]],
        on="row_id",
        how="inner"
    )
    .drop_duplicates(subset=["row_id"])
    .sort_values("row_id")
    .reset_index(drop=True)
)

In [ ]:
# Extract true labels and proxy values as NumPy arrays for evaluation

In [ ]:
y_test = df["preferred_arm"].astype(int).to_numpy()
P_test = df[["p_arm1", "p_arm2", "p_arm3"]].to_numpy(dtype=float)

In [ ]:
# Store the number of test samples and the number of arms

In [ ]:
N = len(y_test)
K = 3

In [ ]:
# Build a policy matrix from DataFrame columns with a given prefix and normalize each row

In [ ]:
def stack_policy(df: pd.DataFrame, prefix: str) -> np.ndarray:
    pi = df[[f"{prefix}arm1", f"{prefix}arm2", f"{prefix}arm3"]].to_numpy(dtype=float)
    return normalize_rows(pi)

In [ ]:
# Stack and normalize all policy probability matrices, including an oracle based on proxy values

In [ ]:
pi_random = stack_policy(df, "pi_random_")
pi_logreg = stack_policy(df, "pi_logreg_")
pi_eps = stack_policy(df, "pi_eps_")
pi_ucb_boot = stack_policy(df, "pi_ucb_boot_")
pi_boot_sample = stack_policy(df, "pi_boot_sample_")
pi_oracle = normalize_rows(P_test)

In [ ]:
# Define a list of policies with their names, probability matrices, and short category labels

In [ ]:
policies = [
    ("Random", pi_random, "baseline"),
    ("Logreg π(x)", pi_logreg, "contextual one-shot"),
    ("ε-greedy(π_logreg)", pi_eps, "contextual one-shot"),
    ("Bootstrap-UCB π(x)", pi_ucb_boot, "contextual one-shot (uncertainty)"),
    ("Bootstrapped posterior sampling π(x)", pi_boot_sample, "contextual one-shot (posterior draw)"),
    ("Oracle π ∝ proxies (SANITY CHECK)", pi_oracle, "sanity"),
]

In [ ]:
# Outputs

In [ ]:
# Define policy names and their corresponding column prefixes for comparison

In [ ]:
policies = [
    ("Logreg π(x)", "pi_logreg_"),
    ("ε-greedy(π_logreg)", "pi_eps_"),
    ("Bootstrap-UCB π(x)", "pi_ucb_boot_"),
    ("Bootstrap posterior draw π(x)", "pi_boot_sample_"),
    ("Random", "pi_random_"),
]

In [ ]:
# Define human-readable labels for the three proxy arms

In [ ]:
arm_labels = ["Proxy 1", "Proxy 2", "Proxy 3"]

In [ ]:
# Extract a single row of policy probabilities for all arms using a column prefix

In [ ]:
def get_pi_row(df, prefix, idx):
    return df.loc[idx, [f"{prefix}arm1", f"{prefix}arm2", f"{prefix}arm3"]].to_numpy(dtype=float)

In [ ]:
# Normalize a vector to sum to one, with safe handling for invalid or zero values

In [ ]:
def normalize(v):
    v = np.asarray(v, dtype=float)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
    s = float(v.sum())
    if s <= 1e-12:
        return np.ones_like(v) / len(v)
    return v / s

In [ ]:
# Compute the L1 distance between two probability vectors


In [ ]:
def l1_distance(p, q):
    return float(np.sum(np.abs(p - q)))

In [ ]:
# Add a small title box inside the top-right corner of a plot

In [ ]:
def title_inside_topright(ax, line1, line2=None, fs=10):
    text = line1 if line2 is None else f"{line1}\n{line2}"
    ax.text(
        0.98, 0.98, text,
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=fs,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=2.5)
    )

In [ ]:
# Prepare plotting data, select the first row, and extract its row ID and true proxy label

In [ ]:
df_plot = df_phase4_test.sort_values("row_id").reset_index(drop=True)
idx = 0
row_id = int(df_plot.loc[idx, "row_id"])
proxy_label = int(df_plot.loc[idx, "preferred_arm"])

In [ ]:
# Create a one-hot reference vector for the true proxy arm, or use uniform if invalid


In [ ]:
p_ref = np.zeros(3, dtype=float)
if 1 <= proxy_label <= 3:
    p_ref[proxy_label - 1] = 1.0
else:
    p_ref[:] = 1 / 3

In [ ]:
# For one selected test sample, build a reference one-hot vector from the true proxy label.
# For each policy, read the three arm probabilities for this sample, normalize them, and compute the L1 distance to the reference.
# Create a 3x2 grid of bar charts: first plot shows the one-hot reference, and the remaining plots show each policy’s distribution.
# Add consistent axis labels, ticks, gridlines, and an in-plot title box that also reports the L1 distance for easy comparison.

In [ ]:
preds = []
for name, prefix in policies:
    p = normalize(get_pi_row(df_plot, prefix, idx))
    d = l1_distance(p, p_ref)
    preds.append((name, p, d))

fig, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(7.2, 10.5),
    dpi=300,
    sharey=True
)
axes = axes.ravel()
x = np.arange(3)

axes[0].bar(x, p_ref)
axes[0].set_xticks(x)
axes[0].set_xticklabels(arm_labels, fontsize=12)
axes[0].set_ylim(0, 1)
axes[0].grid(True, axis="y", alpha=0.25)
title_inside_topright(axes[0], "one-hot proxy label", fs=11)

for i, (name, p, d) in enumerate(preds, start=1):
    ax = axes[i]
    ax.bar(x, p)
    ax.set_xticks(x)
    ax.set_xticklabels(arm_labels, fontsize=12)
    ax.set_ylim(0, 1)
    ax.grid(True, axis="y", alpha=0.25)
    title_inside_topright(ax, name, f"L1 to reference = {d:.3f}", fs=10)


fig.supylabel("Probability mass over proxy dimensions", fontsize=12, x=0.04)

fig.subplots_adjust(
    left=0.115, right=0.98,
    bottom=0.05, top=0.86,
    wspace=0.08, hspace=0.30
)

plt.show()

In [ ]:
# Convert input to a numeric array, clean invalid values, and normalize each row to sum to one

In [ ]:
def normalize_rows(P):
    P = np.asarray(P, dtype=float)
    P = np.nan_to_num(P, nan=0.0, posinf=0.0, neginf=0.0)
    s = np.clip(P.sum(axis=1, keepdims=True), 1e-12, None)
    return P / s

In [ ]:
# Compute entropy for each row to measure how spread out the probability distribution is

In [ ]:
def entropy_rows(P):
    P = np.clip(P, 1e-12, 1.0)
    return -np.sum(P * np.log(P), axis=1)

In [ ]:
# Compute row-wise L1 distance between two probability matrices

In [ ]:
def l1_rows(P, Q):
    return np.sum(np.abs(P - Q), axis=1)

In [ ]:
# Build a reference DataFrame with row ID and proxy values, aligned to the test set

In [ ]:
ref_cols = ["row_id", "p_arm1", "p_arm2", "p_arm3"]
df_ref = (
    df_phase4_test[["row_id"]]
    .merge(df_test_p3[ref_cols], on="row_id", how="inner")
    .drop_duplicates("row_id")
    .sort_values("row_id")
    .reset_index(drop=True)
)


In [ ]:
# Align the test DataFrame to reference row IDs, remove duplicates, and ensure consistent ordering

In [ ]:
df_test_aligned = (
    df_phase4_test.merge(df_ref[["row_id"]], on="row_id", how="inner")
    .drop_duplicates("row_id")
    .sort_values("row_id")
    .reset_index(drop=True)
)

In [ ]:
# Extract proxy values from the reference DataFrame and normalize them into valid probability rows

In [ ]:
P_ref = normalize_rows(df_ref[["p_arm1", "p_arm2", "p_arm3"]].to_numpy(dtype=float))

In [ ]:
# For each policy, verify required probability columns exist, then build a normalized (N x 3) matrix of arm probabilities.
# Compute two structural diagnostics per test sample: (1) entropy to measure how concentrated vs. spread the policy is,
# and (2) L1 distance to a proxy-proportional reference distribution to measure how different the policy’s allocation is.
# Summarize these diagnostics across the full test set with mean and standard deviation, and store them in df_summary.
# Plot two bar charts with error bars: entropy (dispersion) and L1 distance (deviation from proxy structure).

In [ ]:
rows = []
for name, prefix in policies:
    cols = [f"{prefix}arm1", f"{prefix}arm2", f"{prefix}arm3"]
    missing_cols = [c for c in cols if c not in df_test_aligned.columns]
    if missing_cols:
        raise KeyError(f"Missing columns for {name}: {missing_cols}")

    P = normalize_rows(df_test_aligned[cols].to_numpy(dtype=float))

    H = entropy_rows(P)
    D = l1_rows(P, P_ref)

    rows.append({
        "Policy": name,
        "Mean entropy": float(np.mean(H)),
        "Std entropy": float(np.std(H, ddof=1)),
        "Mean L1 distance to proxy": float(np.mean(D)),
        "Std L1 distance": float(np.std(D, ddof=1)),
        "N": int(len(P)),
    })

df_summary = pd.DataFrame(rows)
fig, axes = plt.subplots(2, 1, figsize=(11, 8.5), dpi=140)

x = np.arange(len(df_summary))

# Entropy
axes[0].bar(x, df_summary["Mean entropy"], yerr=df_summary["Std entropy"], capsize=4)
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_summary["Policy"], rotation=20, ha="right")
axes[0].set_ylabel("Entropy H(π)")
axes[0].set_title("Distribution dispersion")
axes[0].grid(True, axis="y", alpha=0.25)

# L1 distance
axes[1].bar(x, df_summary["Mean L1 distance to proxy"], yerr=df_summary["Std L1 distance"], capsize=4)
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_summary["Policy"], rotation=20, ha="right")
axes[1].set_ylabel("L1 distance to proxy-proportional reference")
axes[1].set_title("Structural deviation from proxy")
axes[1].grid(True, axis="y", alpha=0.25)

fig.suptitle(
    "Summary of Probability Allocation Structures (Test set)\n"
    "Note: all references are proxy-proportional and internal",
    fontsize=11
)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()



In [ ]:
# Print the summary table for quick reporting and copy/paste into results or appendix.

In [ ]:
print(df_summary.to_string(index=False, float_format=lambda v: f"{v:.3f}"))